# DSPy RAG — Declarative Retrieval-Augmented Generation
## Declare What You Want — DSPy Optimizer vs. Manual Prompting
⏱ ~15 min

**DSPy** turns RAG from a prompt-engineering exercise into a *compiled program*. Instead of hand-writing a retrieval prompt, you declare **Signatures** (typed input/output fields), compose them into **Modules**, and let an **Optimizer** (Teleprompter) automatically tune the prompts and few-shot examples against a metric. The result: better answers, reproducibly, without touching prompt text.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why DSPy? Declarative vs imperative prompting |
| 2 | **Signatures** — Typed task declarations |
| 3 | **Modules** — Composable predictors (Predict, ChainOfThought) |
| 4 | **Retrieval** — Keyword retriever + wiring context into a Module |
| 5 | **Optimizers** — BootstrapFewShot and MIPROv2 |
| 6 | **Base vs Compiled** — Inspecting what the optimizer changed |
| 7 | **Debugging** — Inspecting prompts, traces, and failure modes |
| ★ | **Advanced** — Saving compiled modules, MIPROv2, multi-hop |

---

### Prerequisites
- Python 3.10+ with `dspy-ai`, `python-dotenv` (or Google Colab)
- An `OPENAI_API_KEY` set in `.env` or Colab Secrets

### Key References
> Khattab, O., Singhvi, A., Maheshwari, P., Zhang, Z., Santhanam, K., Vardhamanan, S., Haq, I., Sharma, A., Jain, T., Moazam, A., Miller, H., Zaharia, M., Potts, C. (2023). *DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines.* https://arxiv.org/abs/2310.03714
>
> Khattab, O., Potts, C., Zaharia, M. (2022). *Demonstrate-Search-Predict: Composing Retrieval and Language Models for Knowledge-Intensive NLP.* https://arxiv.org/abs/2212.14024
>
> DSPy documentation: https://dspy.ai
>
> DSPy GitHub: https://github.com/stanfordnlp/dspy

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "dspy-ai",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
import dspy
from dspy.teleprompt import BootstrapFewShot

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid. "
    "Local: add to .env.  Colab: Secrets panel -> Add 'OPENAI_API_KEY'."
)
print(f"OPENAI_API_KEY present: True  ({key[:8]}...)")

---

## Part 1 — What Is DSPy and Why Does It Exist?

---

### The Prompt Engineering Problem

Classic RAG pipelines require *hand-writing* every prompt. When you change the model, the dataset, or the task, the prompt needs manual re-tuning. This is:
- **Fragile** — small wording changes can flip answer quality
- **Non-transferable** — a prompt tuned for GPT-4o often degrades on GPT-4o-mini
- **Unscalable** — multi-hop pipelines require tuning every step in coordination

DSPy solves this by separating *what* the task is (the Signature) from *how* to prompt for it (handled by the optimizer).

---

### Three Approaches Compared

| Approach | Who writes the prompt? | Adapts to new model? | Few-shots? |
|----------|----------------------|----------------------|------------|
| **Raw prompts** | You, by hand | No — rewrite everything | You add by hand |
| **LangChain LCEL** | You, via `PromptTemplate` | No — rewrite template | You add by hand |
| **DSPy** | Optimizer, from your Signature | Yes — recompile | Auto-bootstrapped |

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Signature** | A class declaring typed input/output fields that define the task |
| **Module** | A composable Python class holding one or more DSPy predictors |
| **Predictor** | A built-in module: `Predict`, `ChainOfThought`, `ReAct`, etc. |
| **Optimizer** (Teleprompter) | Compiles a Module by tuning prompts against a metric |
| **Trainset** | A small labeled dataset (10-50 examples) used by the optimizer |
| **Metric** | A Python function `(example, prediction, trace) -> bool` |
| **Bootstrap** | Running the Module on trainset examples to collect working traces |
| **Compiled Module** | A Module whose prompts and few-shots have been auto-tuned |

### DSPy Compilation Pipeline

```
YOU WRITE                                   OPTIMIZER PRODUCES
───────────────────────────────────         ─────────────────────────────────────────

  class GenerateAnswer(dspy.Signature):
      "Answer a question given context."   <- docstring = instruction
      context:  str = dspy.InputField()    <- typed input
      question: str = dspy.InputField()    <- typed input
      answer:   str = dspy.OutputField()   <- typed output
           |
           |   dspy.ChainOfThought(GenerateAnswer)
           |             |
           v             v
      ┌───────────────────────┐
      │   RAG(dspy.Module)    │   forward():  retrieve -> generate
      └──────────┬────────────┘
                 |
                 |   BootstrapFewShot(metric).compile(RAG(), trainset)
                 |
                 v
      ┌───────────────────────────────────────────────────────┐
      │  Compiled RAG                                         │
      │                                                       │
      │  System: "Answer a question given context."           │
      │  Demo 1: Q: "Who created DSPy?"  A: "Stanford NLP"   │  <- auto-added
      │  Demo 2: Q: "What is Bootstrap?" A: "selects ..."    │  <- auto-added
      │  Current: Q: {question}   Context: {context}         │
      └───────────────────────────────────────────────────────┘
```

> The optimizer *runs* your module on the trainset, collects traces where the metric passes, and uses those traces as automatically-selected few-shot demonstrations. You never write a single demonstration by hand.

---

## Part 2 — Signatures: Typed Task Declarations

---

A **Signature** is a Python class that declares:
- **What inputs the task receives** (`dspy.InputField`)
- **What outputs the task should produce** (`dspy.OutputField`)
- **What the task is** (the class docstring — becomes the instruction in the prompt)

DSPy translates this class directly into a prompt structure. The class docstring becomes the system instruction; each field becomes a labeled slot in the prompt.

---

### Signatures vs PromptTemplate

| Concern | `dspy.Signature` | `PromptTemplate` (LangChain) |
|---------|-----------------|------------------------------|
| What you write | Field names + types + docstring | Full prompt string with `{placeholders}` |
| Prompt wording | Generated and tuned by optimizer | Fixed — you write it manually |
| Few-shot examples | Auto-bootstrapped | You write them |
| Reusable across models | Yes — recompile | No — often model-specific wording |
| Composable in chains | Yes — `dspy.Module.forward()` | Yes — LCEL pipe `|` |

---

### Field Descriptors

Both `InputField` and `OutputField` accept a `desc=` argument. This description is embedded directly into the prompt to constrain the model's behavior.

```python
answer: str = dspy.OutputField(desc="1-2 sentences, factual and concise")
#                                    ^ becomes part of the output instruction
```

In [ ]:
# ─── 2-A: Define Signatures and inspect them ──────────────────────────────────

# Configure DSPy with the LM. This must happen before any dspy.Signature is used.
# temperature=0 for deterministic output — raise to ~0.7 for creative/diverse answers
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini", temperature=0))


class GenerateAnswer(dspy.Signature):
# A Signature is a typed task declaration — docstring becomes the system instruction,
# field names + desc become the prompt template. The optimizer can rewrite both.
    """Answer a question given relevant context from a knowledge base."""

    context: str = dspy.InputField(desc="relevant facts from the knowledge base")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="1-2 sentences, factual and concise")


# Inspect the signature fields
print("Signature: GenerateAnswer")
print(f"  Instructions: {GenerateAnswer.__doc__}")
print()
print("  Input fields:")
for name, field in GenerateAnswer.input_fields.items():
    desc = field.json_schema_extra.get("desc", "(no desc)") if field.json_schema_extra else "(no desc)"
    print(f"    {name}: {desc}")
print()
print("  Output fields:")
for name, field in GenerateAnswer.output_fields.items():
    desc = field.json_schema_extra.get("desc", "(no desc)") if field.json_schema_extra else "(no desc)"
    print(f"    {name}: {desc}")

In [ ]:
# ─── 2-B: Compare Signature variants ─────────────────────────────────────────
# The docstring is the instruction. Changing it changes the model's behavior
# WITHOUT rewriting a prompt template string.


class GenerateAnswerBrief(dspy.Signature):
    """Answer in one short sentence only."""

    context: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class GenerateAnswerWithCitation(dspy.Signature):
    """Answer the question and cite the relevant fact from the context verbatim."""

    context: str = dspy.InputField(desc="facts from the knowledge base")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="answer with a verbatim quote from the context")


DEMO_CONTEXT = (
    "DSPy was created by Stanford NLP and released in 2023. "
    "It replaces hand-crafted prompts with compiled, auto-optimized signatures."
)
DEMO_QUESTION = "Who created DSPy and when?"

for sig_cls in [GenerateAnswer, GenerateAnswerBrief, GenerateAnswerWithCitation]:
    # dspy.Predict wraps a Signature into a callable predictor — no prompt string needed
    pred = dspy.Predict(sig_cls)
    result = pred(context=DEMO_CONTEXT, question=DEMO_QUESTION)
    print(f"Signature: {sig_cls.__name__}")
    print(f"  Instruction: '{sig_cls.__doc__.strip()}'")
    print(f"  Answer:      {result.answer}")
    print()

---

## Part 3 — Modules: Composable Predictors

---

A **Module** is a Python class that:
1. Inherits from `dspy.Module`
2. Declares one or more **predictors** in `__init__` (e.g. `dspy.ChainOfThought`)
3. Implements `forward()` — the execution logic that calls those predictors

Modules are composable: a `RAG` module can call a `Summarize` module whose output feeds a `GenerateAnswer` module. The optimizer can tune all of them at once.

---

### Built-in Predictors

| Predictor | What it adds to the prompt | Use when |
|-----------|--------------------------|----------|
| `dspy.Predict` | Just the Signature fields | Simple input -> output tasks |
| `dspy.ChainOfThought` | Adds a `reasoning` field before the output | Multi-step logic needed |
| `dspy.ReAct` | Alternates Thought / Action / Observation | Tool-calling / agentic tasks |
| `dspy.ProgramOfThought` | Generates and executes code | Math, data manipulation |
| `dspy.MultiChainComparison` | Generates multiple chains, picks best | High-stakes accuracy |

In this workshop we use `ChainOfThought` — it adds a `reasoning` field that the model fills before producing the `answer`, improving accuracy on factual questions.

In [ ]:
# ─── 3-A: Predict vs ChainOfThought ──────────────────────────────────────────
# Predict: direct input -> output
# ChainOfThought: input -> reasoning -> output  (extra intermediate field)

predictor_plain = dspy.Predict(GenerateAnswer)
# ChainOfThought injects an extra 'reasoning' output field — same Signature, richer prompt
predictor_cot = dspy.ChainOfThought(GenerateAnswer)

SAMPLE_CONTEXT = (
    "DSPy Signatures define typed input and output fields using Python class syntax. "
    "DSPy ChainOfThought adds step-by-step reasoning before producing the final answer. "
    "DSPy BootstrapFewShot automatically selects few-shot examples from a labeled trainset."
)
SAMPLE_QUESTION = "What is the difference between Predict and ChainOfThought in DSPy?"

result_plain = predictor_plain(context=SAMPLE_CONTEXT, question=SAMPLE_QUESTION)
result_cot = predictor_cot(context=SAMPLE_CONTEXT, question=SAMPLE_QUESTION)

print("dspy.Predict:")
print(f"  answer: {result_plain.answer}")
print()
print("dspy.ChainOfThought:")
print(f"  reasoning: {result_cot.reasoning}")
print(f"  answer:    {result_cot.answer}")

In [ ]:
# ─── 3-B: Building a minimal Module ──────────────────────────────────────────
# A Module wraps one or more predictors and defines how data flows between them.
# The forward() method is called when you invoke the module like a function.


class AnswerBot(dspy.Module):
    """Minimal module: takes context + question, returns an answer."""

    def __init__(self):
        # Declare predictors as instance attributes.
        # The optimizer can tune these during compilation.
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, context: str, question: str):
        # Call the predictor with the inputs.
        # Returns a Prediction object with .answer (and .reasoning for CoT).
        return self.generate(context=context, question=question)


bot = AnswerBot()
result = bot(context=SAMPLE_CONTEXT, question="What does BootstrapFewShot do?")

print("AnswerBot output:")
print(f"  reasoning: {result.reasoning}")
print(f"  answer:    {result.answer}")
print()
print("Fields available on the result object:", list(result.keys()))

---

## Part 4 — Retrieval: Wiring Context into a Module

---

A RAG module performs two steps in `forward()`:
1. **Retrieve** — find relevant documents for the question
2. **Generate** — pass those documents as `context` to the predictor

In this workshop we start with a simple **keyword retriever** (no embeddings, no API calls) so you can focus on the DSPy concepts. Part 8 shows how to plug in a real vector store.

---

### The Knowledge Base

We use a flat list of facts. In production this would be a Chroma / Pinecone / Weaviate collection.

```
DOCS (knowledge base — flat list of strings)
     |
     |  keyword_retrieve(question, k=3)
     |  -- score each doc by word overlap with question --
     v
  top-k docs joined as a single context string
     |
     v
  ChainOfThought(GenerateAnswer)
     |
     v
  .reasoning  +  .answer
```

In [ ]:
# ─── 4-A: Knowledge base and keyword retriever ───────────────────────────────

DOCS = [
    "DSPy was created by Stanford NLP and released in 2023.",
    "DSPy replaces hand-crafted prompts with compiled, auto-optimized signatures.",
    "DSPy Signatures define typed input and output fields using Python class syntax.",
    "DSPy ChainOfThought adds step-by-step reasoning before producing the final answer.",
    "DSPy BootstrapFewShot automatically selects few-shot examples from a labeled trainset.",
    "MIPROv2 is a DSPy optimizer that jointly tunes instructions and few-shot examples.",
    "DSPy modules are composable Python classes that inherit from dspy.Module.",
    "LangGraph was created by LangChain Inc. and released in January 2024.",
    "LangChain LCEL uses the | pipe operator to chain prompts, models, and parsers.",
    "LangGraph StateGraph compiles into a runnable that can be invoked or streamed.",
]


# k=3 balances recall vs. prompt length — more docs = more context noise for the LM
# Production swap: replace with a vector retriever (e.g. FAISS, Chroma) for semantic search
def keyword_retrieve(question: str, k: int = 3) -> str:
    """TF-style keyword retriever — score docs by word overlap with the question."""
    words = set(question.lower().split())
    scored = [(sum(w in doc.lower() for w in words), doc) for doc in DOCS]
    scored.sort(reverse=True)
    top = [doc for score, doc in scored[:k] if score > 0] or DOCS[:k]
    return "\n".join(top)


# Demonstrate the retriever
test_questions = [
    "Who created DSPy?",
    "What is MIPROv2?",
    "How does LangGraph compile?",
]

for q in test_questions:
    ctx = keyword_retrieve(q)
    print(f"Q: {q}")
    print("Retrieved context:")
    for line in ctx.split("\n"):
        print(f"  - {line}")
    print()

In [ ]:
# ─── 4-B: Full RAG Module ─────────────────────────────────────────────────────
# The RAG module combines retrieval and generation in a single forward() call.


class RAG(dspy.Module):
    def __init__(self):
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question: str):
        # Step 1: retrieve relevant documents
        context = keyword_retrieve(question)
        # Step 2: generate an answer using the retrieved context
        return self.generate(context=context, question=question)


# Test the base (uncompiled) RAG module
# 'base_rag' has no demos injected yet — this is pre-compilation behavior
base_rag = RAG()

sample_questions = [
    "Who created DSPy and when?",
    "How does DSPy differ from LangChain LCEL?",
    "What is the role of MIPROv2 in DSPy?",
]

print("Base RAG (uncompiled) — no few-shot examples\n")
for q in sample_questions:
    result = base_rag(question=q)
    print(f"Q: {q}")
    print(f"  A: {result.answer}")
    print()

---

## Part 5 — Optimizers: Compiling the Pipeline

---

An **Optimizer** (formerly called a Teleprompter) takes an uncompiled Module and a labeled trainset, runs the module on those examples, scores the outputs with a metric function, and returns a new compiled Module with tuned prompts.

---

### Optimizer Comparison

| Optimizer | What it tunes | Training data needed | Speed | Best for |
|-----------|--------------|----------------------|-------|----------|
| `BootstrapFewShot` | Few-shot examples only | 10-50 labeled examples | Fast | Getting started, small trainsets |
| `BootstrapFewShotWithRandomSearch` | Few-shots + random search over configs | 20-100 examples | Medium | Better than basic Bootstrap |
| `MIPROv2` | Instructions + few-shots jointly | 50-200 examples | Slow | Production pipelines, max quality |
| `COPRO` | Instructions only | Few examples | Fast | When you have no labeled data |
| `KNNFewShot` | Retrieval-based few-shot selection | 20+ examples | Fast | When diversity matters |

---

### The Metric Function

The metric is a Python function with this signature:

```python
def my_metric(example: dspy.Example, pred: dspy.Prediction, trace=None) -> bool:
    # example  -- the labeled trainset item (has .question, .answer, etc.)
    # pred     -- the module's output (has .answer, .reasoning, etc.)
    # trace    -- internal trace (None during normal evaluation)
    # Return True if the prediction is good, False otherwise.
    return pred.answer.lower() == example.answer.lower()
```

The optimizer only keeps bootstrapped examples where the metric returns `True`.

---

### BootstrapFewShot — Step by Step

```
1. For each example in trainset:
      run Module.forward(example.inputs)
      if metric(example, output) is True:
          save the full trace (inputs + reasoning + answer)

2. Select up to max_bootstrapped_demos saved traces
3. Inject those traces as few-shot demonstrations into the compiled Module's prompt
4. Return compiled Module
```

In [ ]:
# ─── 5-A: Build the trainset ─────────────────────────────────────────────────
# Each Example declares the inputs (via .with_inputs()) and the expected output.
# The optimizer uses these to evaluate whether bootstrapped demos are good.

# .with_inputs() tells DSPy which fields are inputs vs. expected outputs (labels)
# The optimizer runs the module on inputs, then scores the prediction against the label
TRAINSET = [
    dspy.Example(
        question="Who created DSPy?",
        answer="Stanford NLP",
    ).with_inputs("question"),
    dspy.Example(
        question="What does DSPy ChainOfThought do?",
        answer="adds step-by-step reasoning before the final answer",
    ).with_inputs("question"),
    dspy.Example(
        question="When was LangGraph released?",
        answer="January 2024",
    ).with_inputs("question"),
    dspy.Example(
        question="What is BootstrapFewShot?",
        answer="selects few-shot examples automatically from a trainset",
    ).with_inputs("question"),
    dspy.Example(
        question="What does MIPROv2 optimize?",
        answer="instructions and few-shot examples jointly",
    ).with_inputs("question"),
]

print(f"Trainset: {len(TRAINSET)} labeled examples")
for ex in TRAINSET:
    print(f"  Q: {ex.question}")
    print(f"  A: {ex.answer}")
    print()

In [ ]:
# ─── 5-B: Define the metric and compile with BootstrapFewShot ─────────────────


def validate_answer(example, pred, trace=None) -> bool:
    """Accept answers that are non-empty and at least 10 characters."""
    return bool(pred.answer and len(pred.answer.strip()) > 10)


print("Compiling RAG with BootstrapFewShot...")
print("(This runs the module on the trainset -- expect a few LLM calls)\n")

# BootstrapFewShot: runs forward() on trainset, saves traces that pass the metric as demos
# max_bootstrapped_demos caps demos from successful traces; max_labeled_demos caps raw labels
optimizer = BootstrapFewShot(
    metric=validate_answer,
    max_bootstrapped_demos=2,   # max demos from bootstrap traces
    max_labeled_demos=2,        # max demos taken directly from trainset labels
)

compiled_rag = optimizer.compile(RAG(), trainset=TRAINSET)

# compile() runs the module on trainset, collects passing traces, and bakes them into the module
# After this, compiled_rag.generate.demos will hold the selected few-shot examples
print("Compilation complete.")
print("Base RAG and compiled RAG are both ready for comparison.")

---

## Part 6 — Base vs Compiled: What Changed?

---

After compilation, the module's internal predictor carries **automatically-selected few-shot demonstrations**. These are real (question, reasoning, answer) traces from the trainset where the metric passed. The compiled module passes these into the prompt at inference time — the model sees them as worked examples.

The base module has no demonstrations and relies solely on the Signature's docstring as an instruction.

In [ ]:
# ─── 6-A: Compare base vs compiled answers ───────────────────────────────────

EVAL_QUESTIONS = [
    "Who created DSPy and when?",
    "How does DSPy differ from LangChain LCEL?",
    "What is the role of MIPROv2 in DSPy?",
    "What does ChainOfThought add to a DSPy module?",
]

print("Base RAG vs Compiled RAG\n")
for q in EVAL_QUESTIONS:
    base_result = base_rag(question=q)
    compiled_result = compiled_rag(question=q)
    print(f"Q: {q}")
    print(f"  [base]     {base_result.answer}")
    print(f"  [compiled] {compiled_result.answer}")
    print()

print("Observe: compiled answers tend to be more precise, matching the style")
print("of the few-shot demonstrations the optimizer selected.")

In [ ]:
# ─── 6-B: Inspect what the optimizer actually added ───────────────────────────
# After compilation, compiled_rag.generate.demos contains the bootstrapped demos.
# These are the examples the optimizer chose to include in the prompt.

# .demos is where the optimizer stores its selected few-shot examples —
# inspect this to understand exactly what gets injected into the compiled prompt
demos = compiled_rag.generate.demos
print(f"Bootstrapped demos in compiled RAG: {len(demos)}")
print()

for i, demo in enumerate(demos, 1):
    print(f"Demo {i}:")
    # Demos can be Prediction objects or plain dicts depending on DSPy version
    q = getattr(demo, "question", None) or (demo.get("question") if hasattr(demo, "get") else "?")
    r = getattr(demo, "reasoning", None) or (demo.get("reasoning") if hasattr(demo, "get") else None)
    a = getattr(demo, "answer", None) or (demo.get("answer") if hasattr(demo, "get") else "?")
    print(f"  question:  {q}")
    if r:
        print(f"  reasoning: {str(r)[:80]}...")
    print(f"  answer:    {a}")
    print()

---

## Part 7 — Debugging: Inspecting Prompts and Traces

---

### Common DSPy Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Poor retrieval** | Wrong context -> hallucinated answer | Keyword mismatch | Use embeddings retriever |
| **Short answers** | Metric rejects all bootstrapped demos | `desc=` too restrictive | Loosen output field desc |
| **No demos compiled** | `compiled_rag.generate.demos` is empty | Metric too strict | Relax metric threshold |
| **Metric never True** | Trainset answers don't match model output | Label format mismatch | Normalize both in metric |
| **Reasoning ignored** | ChainOfThought doesn't help | Model too small | Switch to gpt-4o |

---

### Debugging Checklist

1. **Print the last LM call** — `dspy.inspect_history(n=1)`
2. **Check the retrieved context** — call `keyword_retrieve(question)` directly
3. **Check demos after compilation** — `compiled_rag.generate.demos`
4. **Trace a single example** — run one example and inspect `.reasoning`
5. **Relax the metric** — accept any non-empty answer to confirm bootstrap works

In [ ]:
# ─── 7-A: Inspect the last prompt sent to the LM ─────────────────────────────
# dspy.inspect_history() shows the raw prompt + completion for recent calls.
# This is the single most useful debugging tool in DSPy.

# Run a query first to populate the history
_ = compiled_rag(question="What does BootstrapFewShot do?")

print("=" * 60)
print("Last prompt sent to the LM (compiled RAG):")
print("=" * 60)
dspy.inspect_history(n=1)

In [ ]:
# ─── 7-B: Experiment — effect of metric strictness on demo count ───────────────
# If your metric is too strict, the optimizer finds no valid demos.
# If it is too lenient, poor examples get selected as demonstrations.


def strict_metric(example, pred, trace=None) -> bool:
    """Exact match — very strict."""
    return pred.answer.strip().lower() == example.answer.strip().lower()


def lenient_metric(example, pred, trace=None) -> bool:
    """Any non-empty answer passes."""
    return bool(pred.answer and pred.answer.strip())


metrics = [
    (strict_metric, "strict (exact match)"),
    (lenient_metric, "lenient (non-empty)"),
    (validate_answer, "workshop (length > 10)"),
]

for metric_fn, label in metrics:
    opt = BootstrapFewShot(metric=metric_fn, max_bootstrapped_demos=3, max_labeled_demos=2)
    comp = opt.compile(RAG(), trainset=TRAINSET)
    n_demos = len(comp.generate.demos)
    print(f"Metric: {label:<30}  ->  {n_demos} demos selected")

---

## Part 8 ★ — Advanced: Saving, MIPROv2, and Real Retrieval

---

### MIPROv2 — Joint Instruction + Few-Shot Tuning

BootstrapFewShot only tunes the few-shot examples. **MIPROv2** (Multi-prompt Instruction PRoposal Optimizer v2) also proposes and selects the *instruction text* -- the docstring equivalent -- via a Bayesian optimization loop.

```python
from dspy.teleprompt import MIPROv2

optimizer = MIPROv2(
    metric=validate_answer,
    auto="light",      # "light" / "medium" / "heavy" -- controls budget
    num_threads=4,
)
compiled = optimizer.compile(
    RAG(),
    trainset=TRAINSET,
    requires_permission_to_run=False,
)
```

Use MIPROv2 when:
- You have 50+ labeled examples
- BootstrapFewShot quality has plateaued
- You need the highest possible accuracy on a specific metric

---

### Real Embeddings Retriever

Replace the keyword retriever with a Chroma-backed vector store (see example 12):

```python
# Wrap a LangChain retriever for use in DSPy
def chroma_retrieve(question: str, k: int = 3) -> str:
    docs = chroma_retriever.invoke(question)  # Chroma retriever from example 12
    return "\n".join(d.page_content for d in docs[:k])

class ChromaRAG(dspy.Module):
    def __init__(self):
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question: str):
        context = chroma_retrieve(question)    # <- real embedding retrieval
        return self.generate(context=context, question=question)
```

---

### Multi-Hop RAG

Some questions require multiple retrieval steps. DSPy handles this naturally because `forward()` is just Python:

```python
class MultiHopRAG(dspy.Module):
    def __init__(self, hops=2):
        self.retrieve_query = dspy.ChainOfThought("question -> search_query")
        self.generate = dspy.ChainOfThought(GenerateAnswer)
        self.hops = hops

    def forward(self, question: str):
        context_parts = []
        q = question
        for _ in range(self.hops):
            refined = self.retrieve_query(question=q)
            context_parts.append(keyword_retrieve(refined.search_query))
            q = refined.search_query
        return self.generate(context="\n".join(context_parts), question=question)
```

In [ ]:
# ─── 8-A: Saving and loading a compiled module ───────────────────────────────
# In production, compile once and save. Load at inference time -- no re-optimization.

import json
import pathlib

SAVE_PATH = pathlib.Path("./compiled_rag.json")

# Save
compiled_rag.save(str(SAVE_PATH))
print(f"Saved compiled RAG to: {SAVE_PATH}")
print(f"File size: {SAVE_PATH.stat().st_size:,} bytes")

# Load into a fresh module instance
loaded_rag = RAG()
loaded_rag.load(str(SAVE_PATH))

# Verify it works identically to the original compiled module
test_q = "Who created DSPy and when?"
original_answer = compiled_rag(question=test_q).answer
loaded_answer = loaded_rag(question=test_q).answer

print()
print(f"Original: {original_answer}")
print(f"Loaded:   {loaded_answer}")
print(f"Match:    {original_answer == loaded_answer}")

# Peek at the saved JSON structure
with open(SAVE_PATH) as f:
    saved = json.load(f)
print()
print("Top-level keys in saved file:", list(saved.keys()))

---

## Exercises

---

### Exercise 1 — Change the Signature Description

Modify `GenerateAnswer` to return a **bullet-point list** of facts instead of a sentence. Update `desc=` on the output field. Run the base module (no recompile). Does the output format change?

**Hint:** The `desc=` string is injected directly into the prompt. You do not need to touch any prompt template.

### Exercise 2 — Inspect the Extended Signature

After compilation, print `compiled_rag.generate.extended_signature` to see the full prompt DSPy constructs, including the auto-added `reasoning` field from `ChainOfThought`. What do the field names and descriptions look like?

### Exercise 3 — Write a Stricter Metric

Replace `validate_answer` with a metric that checks whether `example.answer` is a substring of `pred.answer` (case-insensitive). Recompile and compare demo count vs the length-based metric. What happens when the metric is very strict?

### Exercise 4 — Extend the Knowledge Base

Add 5 new facts to `DOCS` (e.g., about LangSmith, LangGraph agents, or another framework). Add 2 matching trainset examples. Recompile and verify the new facts are retrievable and appear in compiled answers.

### Exercise 5 — Try MIPROv2

Replace `BootstrapFewShot` with `dspy.teleprompt.MIPROv2(metric=validate_answer, auto="light")` and call `.compile()` with `requires_permission_to_run=False`. Compare the resulting module's demos and instructions to those from BootstrapFewShot.

In [ ]:
# ── Exercise Starters ─────────────────────────────────────────────────────────

# Exercise 1: Change the Signature description
class GenerateAnswerBullets(dspy.Signature):
    """TODO: write a docstring that instructs the model to return bullet points."""

    context: str = dspy.InputField(desc="relevant facts from the knowledge base")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="TODO: describe the desired bullet-point format")


# Test Exercise 1 here (uncomment):
# pred = dspy.Predict(GenerateAnswerBullets)
# result = pred(context="\n".join(DOCS[:3]), question="What are the key facts about DSPy?")
# print(result.answer)


# Exercise 3: Stricter metric
def substring_metric(example, pred, trace=None) -> bool:
    """TODO: return True if example.answer appears (case-insensitive) in pred.answer."""
    pass  # TODO: implement


# Exercise 4: Extended knowledge base
EXTENDED_DOCS = DOCS + [
    # TODO: add 5 new facts here
    "TODO: add your own fact about LangSmith, LangGraph agents, or another framework.",
]

EXTENDED_TRAINSET = TRAINSET + [
    # TODO: add 2 new trainset examples matching your new DOCS
    # dspy.Example(question="...", answer="...").with_inputs("question"),
]

# Exercise 5: MIPROv2
# from dspy.teleprompt import MIPROv2
# mipro_optimizer = MIPROv2(metric=validate_answer, auto="light")
# mipro_compiled = mipro_optimizer.compile(RAG(), trainset=TRAINSET, requires_permission_to_run=False)
# print(mipro_compiled.generate.demos)

print("Exercise starters loaded. Uncomment the relevant lines to run each exercise.")

---

## What's Next?

You now have a working DSPy RAG pipeline that automatically optimizes its own prompts. Here is where to go deeper:

### More RAG patterns in this repo
- **Example 12 — Basic RAG with ChromaDB** (`../12-basic-rag-notebook/rag_workbook.ipynb`): the LangChain LCEL baseline this workshop contrasts against — hand-written prompts, manual few-shots, explicit `PromptTemplate` strings.
- **Example 25 — Adaptive RAG** (`../25-adaptive-rag/`): classify each query at runtime and pick the cheapest retrieval strategy.
- **Example 30 — Agentic RAG** (`../30-agentic-rag/`): LangGraph + RAG for multi-step reasoning with tool use.

### Improve the DSPy pipeline
- **Real embeddings**: replace `keyword_retrieve` with a Chroma-backed retriever (see example 12) for semantic similarity search. The DSPy Module wraps it the same way.
- **MIPROv2**: when you have 50+ labeled examples and want maximum quality, switch from `BootstrapFewShot` to `MIPROv2` — it tunes the instruction text as well as the few-shots.
- **Evaluation harness**: use `dspy.Evaluate` to measure your metric across a held-out dev set before and after compilation to quantify the improvement.

### Further reading
- Khattab et al. (2023). *DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines.* https://arxiv.org/abs/2310.03714
- DSPy RAG tutorial: https://dspy.ai/tutorials/rag/
- DSPy optimizer overview: https://dspy.ai/docs/optimizers/
- DSPy assertions (output constraints at runtime): https://dspy.ai/docs/assertions/

---

*Workshop complete. The compiled module is saved to `./compiled_rag.json` — load it in any new session with `RAG().load("compiled_rag.json")`.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 — Change the Signature Description

**Sample answer:** Setting `desc="bullet-point list of facts, one per line, each starting with -"` on the `answer` field is enough to change the output format without recompilation. The optimizer reinforces this format further by selecting demos that follow it.

**Key insight:** The `desc=` is injected into the prompt directly as a field constraint. You do not need to touch a prompt string anywhere. The Signature field descriptor IS the constraint.

In [ ]:
# Answer Key — Exercise 1
class GenerateAnswerBulletsSolution(dspy.Signature):
    """Answer the question with a concise bullet-point list of facts drawn from the context."""

    context: str = dspy.InputField(desc="relevant facts from the knowledge base")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(
        desc="bullet-point list of facts, one per line, each starting with -"
    )


pred_bullets = dspy.Predict(GenerateAnswerBulletsSolution)
r = pred_bullets(context="\n".join(DOCS[:4]), question="What are the key facts about DSPy?")
print(r.answer)

### Exercise 3 — Write a Stricter Metric

**Sample answer:** A substring metric typically selects fewer demos than the length metric because exact substring matching is harder to satisfy — the model may rephrase the expected answer rather than include it verbatim. If the metric passes on 0 examples, the optimizer falls back to labeled demos from the trainset.

**Key insight:** Metric design is the most important lever in DSPy optimization. Too strict and you get no bootstrapped demos. Too lenient and the demos don't improve quality. Start lenient, then tighten gradually.

In [ ]:
# Answer Key — Exercise 3
def substring_metric_solution(example, pred, trace=None) -> bool:
    """True if the expected answer appears as a substring in the prediction."""
    return example.answer.lower() in pred.answer.lower()


for metric_fn, label in [
    (validate_answer, "workshop length > 10"),
    (substring_metric_solution, "substring match"),
]:
    opt = BootstrapFewShot(metric=metric_fn, max_bootstrapped_demos=3, max_labeled_demos=2)
    comp = opt.compile(RAG(), trainset=TRAINSET)
    print(f"{label:<28} -> {len(comp.generate.demos)} demos selected")

### Exercise 4 — Extend the Knowledge Base

**Sample answer:** New facts are retrievable immediately after adding them to `DOCS` — the keyword retriever is stateless. The optimizer will bootstrap from the new trainset examples and include them as demonstrations if the metric passes.

**What to check:** Call `keyword_retrieve_extended("What is LangSmith?")` directly and verify the new doc appears at the top.

In [ ]:
# Answer Key — Exercise 4
EXTENDED_DOCS_SOLUTION = DOCS + [
    "LangSmith is a platform for debugging and monitoring LangChain applications.",
    "LangSmith provides tracing, evaluation, and dataset management for LLM apps.",
    "DSPy assertions allow modules to enforce output constraints at runtime.",
    "DSPy can be used with any LLM provider, including Anthropic, Google, and local models.",
    "Weaviate is an open-source vector database that supports semantic search at scale.",
]

EXTENDED_TRAINSET_SOLUTION = TRAINSET + [
    dspy.Example(
        question="What is LangSmith used for?",
        answer="debugging and monitoring LangChain applications",
    ).with_inputs("question"),
    dspy.Example(
        question="Can DSPy work with models other than OpenAI?",
        answer="yes, it supports Anthropic, Google, and local models",
    ).with_inputs("question"),
]


def keyword_retrieve_extended(question: str, k: int = 3) -> str:
    words = set(question.lower().split())
    scored = [(sum(w in d.lower() for w in words), d) for d in EXTENDED_DOCS_SOLUTION]
    scored.sort(reverse=True)
    top = [d for _, d in scored[:k] if _ > 0] or EXTENDED_DOCS_SOLUTION[:k]
    return "\n".join(top)


# Quick retrieval check for new facts
for q in ["What is LangSmith?", "Can DSPy use Anthropic models?"]:
    ctx = keyword_retrieve_extended(q)
    print(f"Q: {q}")
    for line in ctx.split("\n"):
        print(f"  - {line}")
    print()